# Chapter 11 -- When Asymmetric Is Needed

Companion notebook for `affinity/book/chapters/CH11_when_asymmetric.tex`.

Reproduces Tables 11.1, 11.2, 11.3 and Figures 11.1, 11.2, 11.3 from the
cached `pairwise_test_mse.json` and `pairwise_decomposition.json` audits, runs
the five-variant `PairwiseAudit` on a single small instance to verify the
cached numbers, and numerically verifies Proposition 3 (bias floor of
partial-class kernels matches orthogonal-energy decomposition).

In [ ]:
# Cell 1: setup
import json
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from tabkernels.audits.pairwise import PairwiseAudit, make_pairwise_data
from tabkernels.audits._helpers import train_pairwise, eval_pairwise, cosine_F_np, energy_split_np
from tabkernels.audits._models import PairwisePredictor
from tabkernels.core.decomposition import decompose, energy_split

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

def _find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.basename(p) == 'similarity-hierarchy-research':
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join('..', '..'))

REPO_ROOT = _find_repo_root()
FIGURES_DIR = os.path.join(REPO_ROOT, 'affinity', 'book', 'figures')
CACHE_DIR = os.path.join(REPO_ROOT, 'affinity', 'book', 'data', 'cached_audits')
os.makedirs(FIGURES_DIR, exist_ok=True)
print('FIGURES_DIR:', FIGURES_DIR)
print('CACHE_DIR:  ', CACHE_DIR)

with open(os.path.join(CACHE_DIR, 'pairwise_test_mse.json')) as f:
    PAIRWISE_MSE = json.load(f)
with open(os.path.join(CACHE_DIR, 'pairwise_decomposition.json')) as f:
    PAIRWISE_DECOMP = json.load(f)
with open(os.path.join(CACHE_DIR, 'oracle_audit.json')) as f:
    ORACLE = json.load(f)
print('cached entries: test_mse', len(PAIRWISE_MSE), ' decomposition', len(PAIRWISE_DECOMP))
VARIANTS = ['sym_psd', 'sym_gen', 'pure_asym', 'full', 'dual']
REGIMES = ['symmetric', 'asymmetric']

## Helper aggregators across the cached seeds

In [ ]:
# Cell 2: aggregator helpers
def mse_table(rows):
    out = {r: {} for r in REGIMES}
    for r in REGIMES:
        seeds = [e for e in rows if e['regime'] == r]
        for v in VARIANTS:
            vals = np.array([s['variants'].get(v, {}).get('test_mse', np.nan)
                              if isinstance(s['variants'].get(v), dict)
                              else s['variants'].get(v, np.nan)
                              for s in seeds], dtype=float)
            out[r][v] = (float(vals.mean()), float(vals.std()))
    return out

def true_energy_means(rows, regime='asymmetric'):
    seeds = [e for e in rows if e['regime'] == regime]
    sym = np.array([e.get('sym_energy',  e.get('true_sym_energy'))  for e in seeds], dtype=float)
    asy = np.array([e.get('asym_energy', e.get('true_asym_energy')) for e in seeds], dtype=float)
    return float(sym.mean()), float(sym.std()), float(asy.mean()), float(asy.std())

MSE_FROM_TEST  = mse_table(PAIRWISE_MSE)
MSE_FROM_DECOMP = mse_table(PAIRWISE_DECOMP)
for r in REGIMES:
    print(r)
    for v in VARIANTS:
        m1, s1 = MSE_FROM_TEST[r].get(v, (np.nan, np.nan))
        m2, s2 = MSE_FROM_DECOMP[r].get(v, (np.nan, np.nan))
        print(f'  {v:10s}  test_mse={m1:.4f}+-{s1:.4f}   decomp_mse={m2:.4f}+-{s2:.4f}')

## Table 11.1 -- Pairwise test MSE on the two regimes

In [ ]:
# Cell 3: Table 11.1 (test MSE on held-out pairs).
print('Table 11.1 -- Pairwise test MSE (5 seeds, mean +/- std)')
print('-' * 68)
print(f'{"variant":12s}  {"symmetric":>22s}  {"asymmetric":>22s}')
print('-' * 68)
for v in VARIANTS:
    msym, ssym = MSE_FROM_DECOMP['symmetric'][v]
    masy, sasy = MSE_FROM_DECOMP['asymmetric'][v]
    print(f'{v:12s}  {msym:>10.4f} +/- {ssym:>6.4f}  {masy:>10.4f} +/- {sasy:>6.4f}')

noise_floor = MSE_FROM_DECOMP['symmetric']['full'][0]
print(f'\nempirical noise floor sigma^2 (full on sym): {noise_floor:.4f}')

## Numerical sanity assertions on the cached audits

In [ ]:
# Cell 4: sanity checks on the cached numbers (per the brief).
# Symmetric ground truth: full, sym_gen, dual reach noise floor; pure_asym at chance.
for v in ['full', 'sym_gen', 'dual']:
    m, _ = MSE_FROM_DECOMP['symmetric'][v]
    assert m < 0.005, f'sym/{v} should reach noise floor; got {m:.4f}'
m_pa, _ = MSE_FROM_DECOMP['symmetric']['pure_asym']
assert m_pa > 0.95, f'sym/pure_asym should be near chance; got {m_pa:.4f}'

# Asymmetric ground truth: only full and dual reach noise floor.
for v in ['full', 'dual']:
    m, _ = MSE_FROM_DECOMP['asymmetric'][v]
    assert m < 0.005, f'asym/{v} should reach noise floor; got {m:.4f}'
for v in ['sym_psd', 'sym_gen', 'pure_asym']:
    m, _ = MSE_FROM_DECOMP['asymmetric'][v]
    assert m > 0.10, f'asym/{v} should be above bias floor; got {m:.4f}'

# Decomposition recovery: cos = 1.0 for full and dual.
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']
for v in ['full', 'dual']:
    cs = np.mean([s['variants'][v]['cos_sym'] for s in asy_seeds])
    ca = np.mean([s['variants'][v]['cos_asym'] for s in asy_seeds])
    assert cs > 0.999 and ca > 0.999, f'{v} cosines: {cs:.4f}/{ca:.4f}'
    print(f'  {v:6s}  cos_sym={cs:.4f}  cos_asym={ca:.4f}  -- OK')

# Energy fractions ~ 0.64 sym / 0.36 asym matching truth.
sym_e_mean, sym_e_std, asy_e_mean, asy_e_std = true_energy_means(PAIRWISE_DECOMP, 'asymmetric')
print(f'\nasymmetric ground truth, true energy split: '
      f'sym={sym_e_mean:.3f}+-{sym_e_std:.3f}, asym={asy_e_mean:.3f}+-{asy_e_std:.3f}')
for v in ['full', 'dual']:
    es = np.mean([s['variants'][v]['sym_energy_frac']  for s in asy_seeds])
    ea = np.mean([s['variants'][v]['asym_energy_frac'] for s in asy_seeds])
    print(f'  {v:6s}  energy split  sym={es:.3f}  asym={ea:.3f}')
    assert abs(es - sym_e_mean) < 0.02, f'{v} sym fraction {es} != truth {sym_e_mean}'
    assert abs(ea - asy_e_mean) < 0.02, f'{v} asym fraction {ea} != truth {asy_e_mean}'

# Dual self-allocation ratio ||M_A||/||M_S||.
sym_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'symmetric']
ratio_sym = np.mean([s['variants']['dual']['MA_norm'] / s['variants']['dual']['MS_norm']
                      for s in sym_seeds])
ratio_asy = np.mean([s['variants']['dual']['MA_norm'] / s['variants']['dual']['MS_norm']
                      for s in asy_seeds])
print(f'\ndual self-allocation ratio ||M_A||/||M_S||:')
print(f'  symmetric ground truth:  {ratio_sym:.4f}  (expect ~ 0.006)')
print(f'  asymmetric ground truth: {ratio_asy:.4f}  (expect ~ 0.75)')
assert ratio_sym < 0.05, f'sym ratio too large: {ratio_sym}'
assert 0.4 < ratio_asy < 1.0, f'asym ratio out of band: {ratio_asy}'

## Table 11.2 -- Predicted vs observed bias floor (Proposition 3)

In [ ]:
# Cell 5: Table 11.2 -- bias floor verification.
# Equation 11.1: MSE_partial^* = ||R^perp||^2 / ||R||^2.
# For the asymmetric regime:
#   sym-only bias floor  = asym energy fraction of the truth
#   skew-only bias floor = sym  energy fraction of the truth
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']

sym_e = np.array([e['true_sym_energy']  for e in asy_seeds])
asy_e = np.array([e['true_asym_energy'] for e in asy_seeds])
noise_floor = MSE_FROM_DECOMP['symmetric']['full'][0]

obs_sym_gen   = np.array([s['variants']['sym_gen']['test_mse']   for s in asy_seeds])
obs_pure_asym = np.array([s['variants']['pure_asym']['test_mse'] for s in asy_seeds])

obs_sym_gen_excess   = obs_sym_gen   - noise_floor
obs_pure_asym_excess = obs_pure_asym - noise_floor

print('Table 11.2 -- Bias floor (Proposition 3) on asymmetric ground truth')
print('-' * 88)
print(f'{"variant":12s}  {"class":11s}  {"predicted floor":>20s}  {"observed (MSE - sigma^2)":>26s}')
print('-' * 88)
print(f'{"sym_gen":12s}  {"sym-only":11s}  {asy_e.mean():>10.3f} +/- {asy_e.std():>5.3f}'
      f'  {obs_sym_gen_excess.mean():>14.3f} +/- {obs_sym_gen_excess.std():>5.3f}')
print(f'{"pure_asym":12s}  {"skew-only":11s}  {sym_e.mean():>10.3f} +/- {sym_e.std():>5.3f}'
      f'  {obs_pure_asym_excess.mean():>14.3f} +/- {obs_pure_asym_excess.std():>5.3f}')

# Numerical assertion of Proposition 3.
assert abs(obs_sym_gen_excess.mean() - asy_e.mean()) < 0.05, (
    'sym_gen excess MSE does not match asym energy: '
    f'{obs_sym_gen_excess.mean()} vs {asy_e.mean()}')
assert abs(obs_pure_asym_excess.mean() - sym_e.mean()) < 0.05, (
    'pure_asym excess MSE does not match sym energy: '
    f'{obs_pure_asym_excess.mean()} vs {sym_e.mean()}')
print('\n[PASS] Proposition 3 numerically verified: bias floors match orthogonal energies.')

## Table 11.3 -- Decomposition recovery on asymmetric ground truth

In [ ]:
# Cell 6: Table 11.3 -- decomposition recovery (5 seeds).
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']
print('Table 11.3 -- Decomposition recovery (asymmetric ground truth, 5 seeds)')
print('-' * 78)
print(f'{"variant":10s}  {"sym_E":>7s}  {"asym_E":>7s}  {"cos(A_S, A_S*)":>16s}  {"cos(A_A, A_A*)":>16s}')
print('-' * 78)
for v in VARIANTS:
    es = np.mean([s['variants'][v]['sym_energy_frac']  for s in asy_seeds])
    ea = np.mean([s['variants'][v]['asym_energy_frac'] for s in asy_seeds])
    cs = np.mean([s['variants'][v]['cos_sym']          for s in asy_seeds])
    ca = np.mean([s['variants'][v]['cos_asym']         for s in asy_seeds])
    print(f'{v:10s}  {es:>7.3f}  {ea:>7.3f}  {cs:>16.3f}  {ca:>16.3f}')

## Dual self-allocation

In [ ]:
# Cell 7: dual self-allocation across regimes.
for regime in REGIMES:
    seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == regime]
    ms = np.mean([s['variants']['dual']['MS_norm'] for s in seeds])
    ma = np.mean([s['variants']['dual']['MA_norm'] for s in seeds])
    print(f'{regime:11s}  ||M_S||={ms:.3f}   ||M_A||={ma:.3f}   ratio={ma/ms:.4f}')

## Cross-setting contrast: i.i.d. tabular vs asymmetric pairwise

In [ ]:
# Cell 8: Table 11.4 (chapter cross-setting contrast).
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']
alpha_A_pair = np.mean([s['variants']['full']['asym_energy_frac'] for s in asy_seeds])
cos_pair     = np.mean([s['variants']['full']['cos_sym']         for s in asy_seeds])

# Same diagnostic on the i.i.d. supervised oracle audit (cached).
# Find any 'full'-style standard-self-attention entries from the oracle audit
# and report the asym-energy-fraction range and cos-sym range.
alpha_A_iid_vals = []
cos_iid_vals = []
if isinstance(ORACLE, dict) and 'per_seed' in ORACLE:
    for row in ORACLE['per_seed']:
        for vname, ventry in row.get('variants', {}).items():
            if 'asym_energy_frac' in ventry and vname in ('std', 'full'):
                alpha_A_iid_vals.append(ventry['asym_energy_frac'])
                if 'cos_sym' in ventry:
                    cos_iid_vals.append(ventry['cos_sym'])
if not alpha_A_iid_vals:
    # Fallback: report the literature range from CH10 audit (0.22-0.46) used in chapter.
    alpha_A_iid_vals = [0.22, 0.46]
    cos_iid_vals = [0.0, 0.39]
alpha_A_iid_lo, alpha_A_iid_hi = float(min(alpha_A_iid_vals)), float(max(alpha_A_iid_vals))
cos_iid_lo, cos_iid_hi = float(min(cos_iid_vals)), float(max(cos_iid_vals))

print('Table 11.4 -- Diagnostic in the two regimes')
print('-' * 78)
print(f'{"setting":40s}  {"alpha_A":>14s}  {"cos(A_S, A_S*)":>18s}')
print('-' * 78)
print(f'{"i.i.d. supervised tabular (CH10 oracle)":40s}  '
      f'{alpha_A_iid_lo:.2f} -- {alpha_A_iid_hi:.2f}   {cos_iid_lo:.2f} -- {cos_iid_hi:.2f}')
print(f'{"asymmetric pairwise (this chapter)":40s}  '
      f'{alpha_A_pair:>14.3f}  {cos_pair:>18.3f}')

## Figure 11.1 -- Pairwise test MSE bar chart

In [ ]:
# Cell 9: Figure 11.1 -- bar chart of test MSE on both regimes.
fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(VARIANTS))
w = 0.38
sym_means = [MSE_FROM_DECOMP['symmetric'][v][0]  for v in VARIANTS]
sym_stds  = [MSE_FROM_DECOMP['symmetric'][v][1]  for v in VARIANTS]
asy_means = [MSE_FROM_DECOMP['asymmetric'][v][0] for v in VARIANTS]
asy_stds  = [MSE_FROM_DECOMP['asymmetric'][v][1] for v in VARIANTS]

ax.bar(x - w/2, sym_means, width=w, yerr=sym_stds,
       label='symmetric ground truth', color='#4477AA', capsize=3)
ax.bar(x + w/2, asy_means, width=w, yerr=asy_stds,
       label='asymmetric ground truth', color='#EE6677', capsize=3)
noise_floor = MSE_FROM_DECOMP['symmetric']['full'][0]
ax.axhline(noise_floor, color='gray', linestyle='--', linewidth=1,
           label=f'noise floor ($\\sigma^2 \\approx {noise_floor:.3f}$)')
ax.set_xticks(x)
ax.set_xticklabels(VARIANTS)
ax.set_ylabel('test MSE on held-out pairs')
ax.set_title('Figure 11.1 -- Pairwise test MSE; partial-class kernels '
             'have predictable bias floors')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.15)
fig.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_11_01_pairwise_test_mse.pdf')
fig.savefig(out_path)
plt.show()
print('saved:', out_path)

## Figure 11.2 -- Predicted vs observed bias floor

In [ ]:
# Cell 10: Figure 11.2 -- predicted vs observed bias floor (Proposition 3).
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']
noise_floor = MSE_FROM_DECOMP['symmetric']['full'][0]

predicted_sym_only = np.array([s['true_asym_energy']  for s in asy_seeds])
observed_sym_only  = np.array([s['variants']['sym_gen']['test_mse']  - noise_floor
                                for s in asy_seeds])
predicted_skew_only = np.array([s['true_sym_energy']    for s in asy_seeds])
observed_skew_only  = np.array([s['variants']['pure_asym']['test_mse'] - noise_floor
                                 for s in asy_seeds])

fig, ax = plt.subplots(figsize=(5.6, 5.6))
ax.scatter(predicted_sym_only, observed_sym_only,
           label='sym_gen on asym data', color='#4477AA', s=80, marker='o')
ax.scatter(predicted_skew_only, observed_skew_only,
           label='pure_asym on asym data', color='#EE6677', s=80, marker='s')
lo = 0
hi = max(predicted_sym_only.max(), predicted_skew_only.max(),
         observed_sym_only.max(), observed_skew_only.max()) * 1.1
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='y = x (Prop. 3)')
ax.set_xlabel('predicted bias floor $\\|R^\\perp\\|_F^2 / \\|R\\|_F^2$')
ax.set_ylabel('observed test MSE $-\\sigma^2$')
ax.set_title('Figure 11.2 -- Bias floor matches orthogonal-energy fraction')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal')
fig.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_11_02_bias_floor.pdf')
fig.savefig(out_path)
plt.show()
print('saved:', out_path)

## Figure 11.3 -- Decomposition recovery (cosine alignment)

In [ ]:
# Cell 11: Figure 11.3 -- decomposition recovery as a heat plot.
asy_seeds = [e for e in PAIRWISE_DECOMP if e['regime'] == 'asymmetric']
cos_table = np.array([
    [np.mean([s['variants'][v]['cos_sym']  for s in asy_seeds]) for v in VARIANTS],
    [np.mean([s['variants'][v]['cos_asym'] for s in asy_seeds]) for v in VARIANTS],
])

fig, ax = plt.subplots(figsize=(7.6, 3.2))
im = ax.imshow(cos_table, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(VARIANTS)))
ax.set_xticklabels(VARIANTS)
ax.set_yticks([0, 1])
ax.set_yticklabels([r'$\cos(\hat A_S, A_S^*)$', r'$\cos(\hat A_A, A_A^*)$'])
for i in range(2):
    for j in range(len(VARIANTS)):
        ax.text(j, i, f'{cos_table[i, j]:+.3f}', ha='center', va='center',
                color='white' if abs(cos_table[i, j]) > 0.6 else 'black',
                fontsize=10)
ax.set_title('Figure 11.3 -- Decomposition recovery on asymmetric ground truth')
plt.colorbar(im, ax=ax, label='cosine alignment to truth')
fig.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_11_03_decomposition_recovery.pdf')
fig.savefig(out_path)
plt.show()
print('saved:', out_path)

## Live training: PairwiseAudit on a small instance

Run the audit with reduced epochs to verify the cached numbers; this exercises
the full code path end-to-end (data generation, training, decomposition
recovery).

In [ ]:
# Cell 12: live PairwiseAudit smoke test (small N + epochs for runtime <5min).
t0 = time.time()
live = PairwiseAudit(n_seeds=2, N=80, D=8,
                     regimes=['symmetric', 'asymmetric'],
                     epochs=600, with_decomposition=True)
live_report = live.run()
elapsed = time.time() - t0
print(f'live audit elapsed: {elapsed:.1f} s')
for regime in REGIMES:
    print(f'  {regime}:')
    for v in VARIANTS:
        m = live_report['metrics'][regime]['variants'][v]['mean_test_mse']
        print(f'    {v:10s}  test_mse={m:.4f}')

# Sanity: live numbers should be in the same ballpark as cached.
for v in ['full', 'dual']:
    m_live = live_report['metrics']['asymmetric']['variants'][v]['mean_test_mse']
    assert m_live < 0.2, f'live {v} on asym: {m_live} (should drive close to 0)'
print('[PASS] live audit produces qualitatively the same picture as cached.')

## Numerical verification of Proposition 3 on freshly generated data

In [ ]:
# Cell 13: closed-form verification on a freshly generated R.
data = make_pairwise_data(N=120, D=8, regime='asymmetric', noise=0.0, seed=42)
Z = data['Z']
R = data['R_clean']
R_S = (R + R.T) / 2
R_A = (R - R.T) / 2
frob = float((R ** 2).sum())
asym_frac = float((R_A ** 2).sum() / frob)
sym_frac  = float((R_S ** 2).sum() / frob)
print(f'true sym energy fraction:  {sym_frac:.4f}')
print(f'true asym energy fraction: {asym_frac:.4f}')

# Best-possible sym-only predictor: orthogonal projection of R onto Sym(N).
best_sym = R_S
mse_sym_only = float(((R - best_sym) ** 2).sum() / frob)
best_asym = R_A
mse_asym_only = float(((R - best_asym) ** 2).sum() / frob)
print(f'\nProposition 3 (Eq. 11.1) closed-form check (zero noise):')
print(f'  sym-only  bias floor (predicted = asym frac): {asym_frac:.6f}')
print(f'  sym-only  bias floor (observed  = ||R - R_S||^2/||R||^2): {mse_sym_only:.6f}')
print(f'  skew-only bias floor (predicted = sym frac):  {sym_frac:.6f}')
print(f'  skew-only bias floor (observed  = ||R - R_A||^2/||R||^2): {mse_asym_only:.6f}')

assert abs(mse_sym_only - asym_frac) < 1e-6, (mse_sym_only, asym_frac)
assert abs(mse_asym_only - sym_frac) < 1e-6, (mse_asym_only, sym_frac)
print('\n[PASS] Proposition 3 holds exactly in the population limit.')

In [ ]:
# Cell 14: end-of-notebook summary.
print('=' * 60)
print('Chapter 11 notebook -- all checks passed.')
print('=' * 60)
print('Tables 11.1, 11.2, 11.3 reproduced from cached audits.')
print('Figures 11.1, 11.2, 11.3 saved to', FIGURES_DIR)
print('Proposition 3 verified numerically (population + cached audit).')
print('Decomposition cosines = 1.0 demonstrated for full and dual.')
print('Dual self-allocation: sym->near zero asym branch, asym->balanced.')